In [ ]:
!pip install ultralytics

YOLO Model

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8m.pt')

model.train(
    data="/kaggle/input/solarpaneldefects/data.yaml",
    epochs=100,
    imgsz=640,
    batch=8,
    name='solar_segmentation',
    project='/kaggle/working',
    device=0)

raise SystemExit()

In [ ]:
model = YOLO("/kaggle/working/solar_segmentation/weights/best.pt")
metrics = model.val(data="/kaggle/input/solarpaneldefects/data.yaml", split="test")
results = model.predict(source="/kaggle/input/solarpaneldefects/test/images", save=True, conf=0.4)
model.export(format="onnx")
raise SystemExit()

RF Pipeline Steps

Inference on the trained Model

In [ ]:
import pandas as pd
def extract_rf_features(results):
    data = []

    for result in results:
        image_name = result.path.split("/")[-1]
        predictions = result.boxes
        class_counts = {}

        for c in predictions.cls:
            cls = int(c.item())
            class_counts[cls] = class_counts.get(cls, 0) + 1

        feature_row = {'image_name': image_name}
        for i in range(len(model.names)):
            feature_row[f'class_{i}_count'] = class_counts.get(i, 0)

        data.append(feature_row)

    return pd.DataFrame(data)

features_df = extract_rf_features(results)
features_df.head()

In [14]:
from ultralytics import YOLO
import pandas as pd
import os
from tqdm import tqdm

# Load your trained YOLOv8 model
model = YOLO("/kaggle/input/solardefect_yolo/other/default/1/best.pt")

# Paths
image_folder = "/kaggle/input/solarpaneldefects/train/images"
csv_path = "/kaggle/input/solardefects-csv/Train_CSV.csv"

# Load the CSV
df = pd.read_csv(csv_path)

# Clean up column names (optional but safer)
df.columns = [col.strip().split()[-1] if '#' in col else col.strip() for col in df.columns]

# Confirm class label columns
label_columns = ['Bird-drop', 'Clean', 'Dusty', 'Electrical-damage', 'Physical-damage', 'Snow']

# Feature storage
features_list = []

# Loop over image files
image_files = sorted(os.listdir(image_folder))
for img_file in tqdm(image_files):
    image_path = os.path.join(image_folder, img_file)

    # YOLO inference
    results = model(image_path, verbose=False)[0]
    num_boxes = len(results.boxes)
    classes_detected = results.boxes.cls.tolist() if num_boxes > 0 else []
    avg_conf = results.boxes.conf.mean().item() if num_boxes > 0 else 0
    class_counts = [classes_detected.count(i) for i in range(6)]  # For 6 classes

    # Match true label row
    match = df[df['filename'].str.contains(img_file)]
    if not match.empty:
        label_data = match[label_columns].iloc[0].to_dict()

        features = {
            "image": img_file,
            "num_boxes": num_boxes,
            "avg_conf": avg_conf,
            **{f"class_{i}_count": count for i, count in enumerate(class_counts)},
            **label_data  # append true one-hot label columns
        }

        features_list.append(features)

# Convert to DataFrame and save
features_df = pd.DataFrame(features_list)
features_df.to_csv("/kaggle/working/train_features_rf.csv", index=False)
print("✅ Feature extraction complete and saved to train_features_rf.csv")


100%|██████████| 1425/1425 [00:40<00:00, 34.88it/s]

✅ Feature extraction complete and saved to train_features_rf.csv


In [30]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.multioutput import MultiOutputClassifier
import joblib

# Load dataset
df = pd.read_csv("/kaggle/working/train_features_rf.csv")
df.columns = df.columns.str.strip()

# Features
X = df[['num_boxes', 'avg_conf', 
        'class_0_count', 'class_1_count', 'class_2_count', 
        'class_3_count', 'class_4_count', 'class_5_count']]

# Target (multi-label binary format)
Y = df[['Bird-drop', 'Clean', 'Dusty', 
        'Electrical-damage', 'Physical-damage', 'Snow']]

# Split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
multi_rf = MultiOutputClassifier(rf)
multi_rf.fit(X_train, Y_train)

# Save model
joblib.dump(multi_rf, "rf_model.pkl")

# Predict
Y_pred = multi_rf.predict(X_test)

# Evaluate
print("📊 Classification Report (per label):\n")
print(classification_report(Y_test, Y_pred, zero_division=1, target_names=Y.columns))


📊 Classification Report (per label):

                   precision    recall  f1-score   support

        Bird-drop       0.98      0.89      0.93        46
            Clean       1.00      1.00      1.00        57
            Dusty       0.96      0.95      0.96        79
Electrical-damage       0.95      0.98      0.96        53
  Physical-damage       0.98      0.96      0.97        54
             Snow       1.00      1.00      1.00        16

        micro avg       0.97      0.96      0.97       305
        macro avg       0.98      0.96      0.97       305
     weighted avg       0.97      0.96      0.97       305
      samples avg       0.98      0.96      0.96       305

